#### Conditional Chains
We will give the comment of the product. if the sentiment of the product is POS, we will give the positive feedback text, else negative feedback text

In [122]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.prompts import PromptTemplate
from langchain_core.globals import set_debug

# Call this at the start of your script
set_debug(False)

In [61]:
load_dotenv()

True

In [114]:
model = ChatOpenAI(
    model='gpt-5.4-mini',
    max_completion_tokens=100
)
parser = StrOutputParser()

In [99]:
prompt = PromptTemplate(
    template="""Give the comment {comment}, Give only the sentiment to Negative or Positive. Do not give other text in the response.""",
    input_variables={'comment'}
)

In [107]:
chain = prompt | model | parser | RunnableLambda(lambda x: {"sentiment": x})

In [108]:
comment = "This phone is so bad."
res = chain.invoke({'comment': comment})

In [109]:
print(res)

{'sentiment': 'Negative'}


In [132]:
# Now we are going to pass the output of llm into branch which then run the chains based on conditions.
pos_prompt_response = PromptTemplate(
    template='Give the positive response to the user as user has provided the positive feedback.',
    input_variables={'comment'}
)

neg_prompt_response = PromptTemplate(
    template='Give the negative response to the user as user has provided the negative feedback.',
    input_variables={'comment'}
)


branch = RunnableBranch(
    (lambda x: x["sentiment"]=='Positive', pos_prompt_response | model | parser),
    lambda x: neg_prompt_response | model | parser
)

In [136]:
final_chain = chain | branch
comment = "This phone is so good."
res = final_chain.invoke({'comment': comment})
print("without parser: " + str(res))


# final_chain_with_parser = final_chain | parser
# res = final_chain_with_parser.invoke({'comment': comment})
# print("\n\n")
# print(res)

without parser: Thank you so much for your positive feedback! We’re glad to hear you had a great experience. Your support means a lot to us, and we look forward to serving you again!
